# Postprocessing PureCLIP peaks without KO input control

This notebook postprocesses significant crosslink positions detected by PureCLIP
for one sample or a pooled group. The default inputs are pooled-WT calls made
without KO input control. Signal on the matching strand is used to position
non-overlapping candidate binding sites with a uniform width of **9 nt**.

The postprocessing approach is adapted from Supplementary Data 2 of Busch et al.,
*iCLIP data analysis: A complete pipeline from sequencing reads to RBP binding
sites*, **Methods 178**, 49–62 (2020).

The steps are:
1. Import plus/minus crosslink tracks and significant PureCLIP positions.
2. Retain standard chromosomes and merge nearby positions separated by up to 7 nt.
3. Center short regions on their signal maximum and extend by 4 nt on each side.
4. Place multiple non-overlapping 9-nt windows in longer regions, in decreasing order of signal at their central nucleotide.
5. Retain candidates containing signal at one or more nucleotides and export BED.

**Inputs:** strand-specific normalized BigWigs and the six-column PureCLIP crosslink-position BED for the same sample/group. **Outputs:** the merged regions (`merged_sites_bed`) and curated candidate sites (`output_bed`). These candidates are passed to analysis 02 for replicate support and genomic annotation; they are not yet the final reproducible or WT-enriched binding-site set.

## Input/output configuration

Edit these paths before running the analysis. Relative paths are resolved from the repository root. Run the cells in order using the R kernel.

`plus_bw`, `minus_bw`, and `pureclip_sites` must describe the same sample or group.
To inspect an individual WT library instead of pooled WT, change all three inputs
and use distinct output filenames. The default `output_bed` is the input expected
by analysis 02. Paths are resolved from the repository root.

In [1]:
params <- list(
  plus_bw = "results/preprocessing/akna/crosslinked_nucleotides/replicates.plus.bw",
  minus_bw = "results/preprocessing/akna/crosslinked_nucleotides/replicates.minus.bw",
  pureclip_sites = "results/preprocessing/akna/peak_calling/replicates_noinputcontroldata_PureCLIP.crosslink_sites_short.bed",
  output_bed = "results/analysis/01_pureclip_postprocessing/replicates_noKO_curated_peaks.bed",
  merged_sites_bed = "results/analysis/01_pureclip_postprocessing/replicates_noKO_sign_sites_max_gap_width_7.bed"
)

project_root <- normalizePath(getwd(), mustWork = TRUE)
if (!file.exists(file.path(project_root, "workflow", "Snakefile"))) {
  project_root <- normalizePath(file.path(project_root, ".."), mustWork = TRUE)
}
if (!file.exists(file.path(project_root, "workflow", "Snakefile"))) {
  stop("Open this notebook from the repository root or its analysis directory.")
}
resolve_repo_path <- function(path) {
  if (grepl("^(/|[A-Za-z]:[/\\\\])", path)) path else file.path(project_root, path)
}
paths <- lapply(params, resolve_repo_path)


## Define input and output files

Resolve the configured filenames and create parent directories for the two BED
exports. Existing input files are read without modification.

In [2]:
plus_bw_file <- paths$plus_bw
minus_bw_file <- paths$minus_bw
peak_file <- paths$pureclip_sites
output_file <- paths$output_bed
sign_sites <- paths$merged_sites_bed

dir.create(dirname(output_file), recursive = TRUE, showWarnings = FALSE)
dir.create(dirname(sign_sites), recursive = TRUE, showWarnings = FALSE)


## Load required libraries

In [3]:
library(GenomicRanges)
library(rtracklayer)
library(ggplot2)


## Import crosslink events and significant positions

The BigWigs provide run-length encoded per-position signal. Minus-strand signal
is converted to its absolute value so maximum finding uses signal magnitude.
BED import converts the file's 0-based, half-open coordinates to the 1-based,
inclusive coordinates used by `GRanges`. Strand information is retained.

In [4]:
plus_cov <- import.bw(plus_bw_file, as = "Rle")
minus_cov <- abs(import.bw(minus_bw_file, as = "Rle"))
peaks <- import(peak_file)
cat("Number of peaks from PureCLIP:", length(peaks), "\n")


Number of peaks from PureCLIP: 22532 


## Filter chromosomes and merge positions separated by up to 7 nt

`keepStandardChromosomes` removes nonstandard sequence names. `reduce` merges
positions on the same strand with `min.gapwidth = 8`, joining gaps smaller than
8 nt. The exported merged regions can have different widths. The lower width
threshold is zero, so all nonempty merged regions proceed to the next steps.

In [5]:
peaks_sc <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
cat("Number of standard chromosome peaks:", length(peaks_sc), "\n")

peaks_sc_m <- reduce(peaks_sc, min.gapwidth = 8)
cat("Number of peaks after merging gaps up to 7 nt:", length(peaks_sc_m), "\n")
export(peaks_sc_m, sign_sites)

width_threshold <- 0
peaks_sc_m_pw <- peaks_sc_m[width(peaks_sc_m) > width_threshold]


Number of standard chromosome peaks: 22511 


Number of peaks after merging gaps up to 7 nt: 20009 


## Center short regions and extend by 4 nt

For each region of width ≤9 nt, find its maximum signal. On the plus strand,
`which.max` selects the first maximum from the left. On the minus strand, the
signal is reversed before finding the maximum, so ties are resolved from the
5′ end on that strand. Extend the selected nucleotide by 4 nt upstream and
downstream to obtain a 9-nt window.

In [6]:
peaks_sc_m_pw_short <- peaks_sc_m_pw[width(peaks_sc_m_pw) <= 9]
peaks_sc_m_pw_short_plus <- peaks_sc_m_pw_short[
  strand(peaks_sc_m_pw_short) == "+"
]
# Identify the maximum-signal nucleotide within each short plus-strand region.
max_pos_plus <- sapply(plus_cov[peaks_sc_m_pw_short_plus], which.max)
peaks_sc_m_pw_short_plus_centered <- GRanges(
  seqnames = seqnames(peaks_sc_m_pw_short_plus),
  ranges = IRanges(
    start = start(peaks_sc_m_pw_short_plus) + max_pos_plus - 1,
    width = 1
  ),
  strand = "+"
)

peaks_sc_m_pw_short_minus <- peaks_sc_m_pw_short[
  strand(peaks_sc_m_pw_short) == "-"
]
# Reverse minus-strand signal to search in the 5′-to-3′ direction.
max_pos_minus <- sapply(
  lapply(minus_cov[peaks_sc_m_pw_short_minus], rev),
  which.max
)
peaks_sc_m_pw_short_minus_centered <- GRanges(
  seqnames = seqnames(peaks_sc_m_pw_short_minus),
  ranges = IRanges(
    start = end(peaks_sc_m_pw_short_minus) - max_pos_minus + 1,
    width = 1
  ),
  strand = "-"
)

# Expanding a single-nucleotide GRanges by 4 produces a 9-nt interval.
peaks_sc_m_pw_sc <- c(
  peaks_sc_m_pw_short_plus_centered + 4,
  peaks_sc_m_pw_short_minus_centered + 4
)
cat("Number of resized short peaks:", length(peaks_sc_m_pw_sc), "\n")


Number of resized short peaks: 19963 


## Fit multiple binding sites in long regions: plus strand

For regions wider than 9 nt, enumerate 9-nt candidate windows centered on each
nucleotide of the merged region. Rank the centers by decreasing crosslink signal,
keep the strongest candidate, and iteratively remove candidates overlapping
selected windows. This allows a broad region to contribute several sites.

The overlap assertions count self-hits: one hit per selected site means there
are no additional same-strand overlaps in the selected set.

In [7]:
peaks_sc_m_pw_long <- peaks_sc_m_pw[width(peaks_sc_m_pw) > 9]
peaks_sc_m_pw_long_plus <- peaks_sc_m_pw_long[
  strand(peaks_sc_m_pw_long) == "+"
]
peaks_sc_m_pw_long_plus_curated <- GRanges()

if (length(peaks_sc_m_pw_long_plus) > 0) {
  for (i in seq_along(peaks_sc_m_pw_long_plus)) {
    peak_tmp <- peaks_sc_m_pw_long_plus[i]
    peak_set <- GRanges(
      seqnames = seqnames(peak_tmp),
      ranges = IRanges(
        start = (start(peak_tmp) - 4):(end(peak_tmp) - 4),
        width = 9
      ),
      strand = strand(peak_tmp)
    )
    # Shrink each 9-nt window by 4 on each side to rank its central nucleotide.
    max_ordered <- order(
      sapply(plus_cov[peak_set - 4], runValue),
      decreasing = TRUE
    )
    # Reorder by signal and iteratively retain only non-overlapping candidates.
    peaks_ord <- peak_set[max_ordered]
    peaks_final <- peaks_ord[1]
    peaks_cands <- peaks_ord[-1]
    peaks_cands <- peaks_cands[
      -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
    ]

    while (length(peaks_cands) > 0) {
      peaks_final <- c(peaks_cands[1], peaks_final)
      peaks_cands <- peaks_cands[-1]
      peaks_cands <- peaks_cands[
        -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
      ]
    }
    peaks_sc_m_pw_long_plus_curated <- c(
      peaks_sc_m_pw_long_plus_curated,
      peaks_final
    )
    stopifnot(
      nrow(as.matrix(findOverlaps(peaks_sc_m_pw_long_plus_curated))) ==
        length(peaks_sc_m_pw_long_plus_curated)
    )
    if (i %% 500 == 0) cat("Plus-strand iteration", i, "complete\n")
  }
}


## Fit multiple binding sites in long regions: minus strand

Apply the same iterative procedure with candidate centers enumerated from high
to low genomic coordinates, following the minus strand's 5′-to-3′ direction.
Only candidates on this strand are compared for overlap.

In [8]:
peaks_sc_m_pw_long_minus <- peaks_sc_m_pw_long[
  strand(peaks_sc_m_pw_long) == "-"
]
peaks_sc_m_pw_long_minus_curated <- GRanges()

if (length(peaks_sc_m_pw_long_minus) > 0) {
  for (i in seq_along(peaks_sc_m_pw_long_minus)) {
    peak_tmp <- peaks_sc_m_pw_long_minus[i]
    # Enumerate candidate windows in minus-strand order.
    peak_set <- GRanges(
      seqnames = seqnames(peak_tmp),
      ranges = IRanges(
        end = (end(peak_tmp) + 4):(start(peak_tmp) + 4),
        width = 9
      ),
      strand = strand(peak_tmp)
    )
    # Rank the central-nucleotide signal before selecting non-overlapping sites.
    max_ordered <- order(
      sapply(minus_cov[peak_set - 4], runValue),
      decreasing = TRUE
    )
    peaks_ord <- peak_set[max_ordered]
    peaks_final <- peaks_ord[1]
    peaks_cands <- peaks_ord[-1]
    peaks_cands <- peaks_cands[
      -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
    ]

    while (length(peaks_cands) > 0) {
      peaks_final <- c(peaks_cands[1], peaks_final)
      peaks_cands <- peaks_cands[-1]
      peaks_cands <- peaks_cands[
        -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
      ]
    }
    peaks_sc_m_pw_long_minus_curated <- c(
      peaks_sc_m_pw_long_minus_curated,
      peaks_final
    )
    stopifnot(
      nrow(as.matrix(findOverlaps(peaks_sc_m_pw_long_minus_curated))) ==
        length(peaks_sc_m_pw_long_minus_curated)
    )
    if (i %% 500 == 0) cat("Minus-strand iteration", i, "complete\n")
  }
}


## Combine, require crosslink support, and export candidate sites

Combine the short-region and long-region windows. A candidate must contain
positive signal at **at least one nucleotide** in its matching-strand track.
This counts occupied positions, not the number of crosslink events at a position.
The final overlap check requires only self-hits. BED export converts coordinates
back to 0-based starts and exclusive ends for analysis 02 and genome browsers.

In [9]:
peaks_sl_curated <- c(
  peaks_sc_m_pw_sc,
  peaks_sc_m_pw_long_plus_curated,
  peaks_sc_m_pw_long_minus_curated
)
stopifnot(
  nrow(as.matrix(findOverlaps(peaks_sl_curated))) == length(peaks_sl_curated)
)

# Minimum number of positions with nonzero crosslink signal in each window.
num_positions_with_crosslinks <- 1
peaks_sl_curated_plus <- peaks_sl_curated[strand(peaks_sl_curated) == "+"]
peaks_curated_plus <- peaks_sl_curated_plus[
  sapply(lapply(plus_cov[peaks_sl_curated_plus], function(x) x > 0), sum) >=
    num_positions_with_crosslinks
]

peaks_sl_curated_minus <- peaks_sl_curated[strand(peaks_sl_curated) == "-"]
peaks_curated_minus <- peaks_sl_curated_minus[
  sapply(lapply(minus_cov[peaks_sl_curated_minus], function(x) x > 0), sum) >=
    num_positions_with_crosslinks
]
peaks_curated <- c(peaks_curated_plus, peaks_curated_minus)

cat("Number of curated plus-strand peaks:", length(peaks_curated_plus), "\n")
cat("Number of curated minus-strand peaks:", length(peaks_curated_minus), "\n")
cat("Total number of curated peaks:", length(peaks_curated), "\n")
stopifnot(
  nrow(as.matrix(findOverlaps(peaks_curated))) == length(peaks_curated)
)
export(peaks_curated, output_file)


Number of curated plus-strand peaks: 9930 


Number of curated minus-strand peaks: 10108 


Total number of curated peaks: 20038 
